In [1]:
#*************************************************************************
#   > Filename    : make_gc_bit_great_again.py
#   > Description : GIN trained on BBBP
#*************************************************************************


## Setup

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile

!unzip quantize_function.zip
!unzip quantize_method.zip
!unzip utils.zip

In [ ]:
print('Installing packages...')

In [ ]:
!pip install torch_geometric
!pip install torch_scatter
!pip install torch_sparse

## Packages and Libraries

In [19]:
import os
import numpy as np
import random
import torch
import torch.nn as nn
from torch.nn import Linear, Sequential, ReLU, Identity, BatchNorm1d as BN
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree,remove_self_loops
from torch_geometric.data import Data
from torch_geometric.datasets import TUDataset,Planetoid,GNNBenchmarkDataset
from torch_geometric.loader import DataLoader
from torch_scatter import scatter_mean
from torch.autograd.function import InplaceFunction
from torch_geometric.nn import GCNConv,GINConv,global_mean_pool,TopKPooling
import torch_geometric.transforms as T


import statistics as stat
from tabulate import tabulate

from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import time
import argparse
#################################################
from quantize_function.u_quant_gc_bit_debug import *
from quantize_function.MessagePassing_gc_bit import GINConvMultiQuant
from quantize_function.get_scale_index import get_deg_index, get_scale_index


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## function to measure model size

In [13]:
def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB



## Function for Quatization

In [2]:
def paras_group(model):
    all_params = model.parameters()
    weight_paras=[]
    quant_paras_bit_weight = []
    quant_paras_bit_fea = []
    quant_paras_scale_weight = []
    quant_paras_scale_fea = []
    quant_paras_scale_xw = []
    quant_paras_bit_xw = []
    other_paras = []
    for name,para in model.named_parameters():
        if('quant' in name and 'bit' in name and 'weight' in name):
            quant_paras_bit_weight+=[para]
            # para.requires_grad = False
        elif('quant' in name and 'bit' in name and 'fea' in name):
            quant_paras_bit_fea+=[para]
        elif('quant' in name and 'bit' not in name and 'weight' in name):
            quant_paras_scale_weight+=[para]
            # para.requires_grad = False
        elif('quant' in name and 'bit' not in name and 'fea' in name):
            quant_paras_scale_fea+=[para]
        elif('xw'in name and 'q' in name and 'bit' not in name):
            quant_paras_scale_xw+=[para]
        elif('xw'in name and 'q' in name and 'bit' in name):
            quant_paras_bit_xw+=[para]
        elif('weight' in name and 'quant' not in name ):
            weight_paras+=[para]
    params_id = list(map(id,quant_paras_bit_fea))+list(map(id,quant_paras_bit_weight))+list(map(id,quant_paras_scale_weight))+list(map(id,quant_paras_scale_fea))+list(map(id,weight_paras))\
    +list(map(id,quant_paras_scale_xw))+list(map(id,quant_paras_bit_xw))
    other_paras = list(filter(lambda p: id(p) not in params_id, all_params))
    return weight_paras,quant_paras_bit_weight,quant_paras_bit_fea,quant_paras_scale_weight,quant_paras_scale_fea,quant_paras_scale_xw,quant_paras_bit_xw,other_paras

def setup_seed(seed):
      torch.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)
      np.random.seed(seed)
      random.seed(seed)
    #  torch.backends.cudnn.deterministic = True

def parameter_stastic(model,dataset,hidden_units):
    w_Byte = 0
    a_Byte = 0
    for name, par in model.named_parameters():
        if(('bit' in name)&('weight' in name)):
            if('conv1' in name):
                scale = dataset.num_node_features
            else:
                scale = hidden_units
            par = torch.floor(par)
            w_Byte = scale*par.sum()/8./1024.+w_Byte
        elif(('bit' in name)&('fea' in name)):
            if('conv1' in name):
                a_scale = 0
            else:
                a_scale = hidden_units
            # a_scale = dataset.data.num_nodes
            par = torch.floor(par)
            a_Byte = a_scale*par.sum()/8./1024.+a_Byte
    return w_Byte, a_Byte

class ResettableSequential(nn.Sequential):
    def reset_parameters(self):
        for child in self.children():
            if hasattr(child, "reset_parameters"):
                child.reset_parameters()
    def forward(self,input,edge_index,bit_sum):
        for model in self:
            input,_,bit_sum = model(input,edge_index,bit_sum)
        return input,bit_sum



In [3]:
# Relu and Batch Normalization
class relu(nn.Module):
    def __init__(self,):
        super().__init__()
    def forward(self,x,edge_index,bit_sum):
        x[x<0] = 0
        return x,edge_index,bit_sum

class bn(nn.Module):
    def __init__(self,hidden_units):
        super().__init__()
        self.bn = nn.BatchNorm1d(hidden_units)
    def forward(self,x,edge_index,bit_sum):
        x = self.bn(x)
        return x,edge_index,bit_sum

## qGIN Model with Quatization

In [4]:

class qGIN(nn.Module):
    def __init__(self, dataset, num_layers, hidden_units, bit, num_deg=1000, is_q=False,
                    uniform=False,init='norm'):
        super(qGIN, self).__init__()
        gin_layer = GINConvMultiQuant
        self.bit = bit
        para_list=[[{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.70,'gama_std':0.1}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.6,'gama_std':0.7}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.76,'gama_std':0.68}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.6,'gama_std':0.5}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1},{'gama_init':0.6,'gama_std':0.3}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1}],
                   [{'alpha_init':0.01,'gama_init':0.01,'alpha_std':0.1,'gama_std':0.1}]]
        if(is_q):
            # As the DQ, we either don't quantize the input features of the REDDIT-BINARY dataset because the feature is only 1-dimension.
            self.conv1 = gin_layer(
                ResettableSequential(
                    QLinear(9,hidden_units, num_deg, bit,para_dict=para_list[0][0], all_positive=True,
                            quant_fea=False,
                            uniform=uniform,init=init),
                    relu(),
                    QLinear(hidden_units, hidden_units, num_deg, bit, para_dict=para_list[0][1],all_positive=True,
                            uniform=uniform,init=init),
                    relu(),
                ),
                train_eps=True,
                in_features=num_deg, out_features=1,
                bit=bit, para_dict=para_list[0][2],quant_fea=False,uniform=uniform
            )
        else:
            self.conv1 = GINConv(
                nn.Sequential(
                    nn.Linear(9, hidden_units),
                    nn.ReLU(),
                    nn.Linear(hidden_units, hidden_units),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_units),
                ),
                train_eps=True,
            )
        self.convs = torch.nn.ModuleList()
        for i in range(num_layers - 1):
            if(is_q):
                self.convs.append(
                    gin_layer(
                        ResettableSequential(
                            QLinear(hidden_units, hidden_units, num_deg,bit, para_dict=para_list[0][0],all_positive=False,
                                    uniform=uniform,init=init),
                            relu(),
                            QLinear(hidden_units, hidden_units, num_deg,bit, para_dict=para_list[0][1], all_positive=True,
                                    uniform=uniform,init=init),
                            relu(),
                        ),
                        train_eps=True,
                        in_features=num_deg, out_features=hidden_units,
                        bit=bit, para_dict=para_list[0][2], uniform=uniform,quant_fea=True
                    )
                )
            else:
                self.convs.append(
                    GINConv(
                        nn.Sequential(
                            nn.Linear(hidden_units, hidden_units),
                            nn.ReLU(),
                            nn.Linear(hidden_units, hidden_units),
                            nn.ReLU(),
                            nn.BatchNorm1d(hidden_units),
                        ),
                        train_eps=True,
                    )
                )
        self.bn_list = torch.nn.ModuleList()
        for i in range(num_layers):
            self.bn_list.append(nn.BatchNorm1d(hidden_units))
        if(is_q):
            self.lin1 = QLinear(hidden_units, hidden_units, num_deg, bit, para_dict=para_list[-1][0], all_positive=False,
                                        uniform=uniform,init=init)
            self.lin2 = QLinear(hidden_units, dataset.num_classes, num_deg, bit, para_dict=para_list[-1][0], all_positive=True,
                                        uniform=uniform,init=init)
        else:
            self.lin1 = nn.Linear(hidden_units, hidden_units)
            self.lin2 = nn.Linear(hidden_units, dataset.num_classes)

    def reset_parameters(self):
        self.conv1.reset_parameters()
        for conv in self.convs:
            conv.reset_parameters()
        self.lin1.reset_parameters()
        self.lin2.reset_parameters()

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        bit_sum=x.new_zeros(1)
        x,bit_sum = self.conv1(x, edge_index,bit_sum)
        x = self.bn_list[0](x)
        # x,_,bit_sum = self.embeding(x,edge_index,bit_sum)
        # x = F.relu(x)
        i = 1
        for conv in self.convs:
            x,bit_sum = conv(x,edge_index,bit_sum)
            x = self.bn_list[i](x)
            i=i+1
        x = global_mean_pool(x, batch)
        x,_,bit_sum = self.lin1(x,edge_index,bit_sum)
        x = F.relu(x)
        x,_,bit_sum = self.lin2(x,edge_index,bit_sum)
        return F.log_softmax(x, dim=-1),bit_sum
    




# Helpful Function

In [5]:
class NormalizedDegree(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, data):
        deg = degree(data.edge_index[0], dtype=torch.float)
        deg = (deg - self.mean) / self.std
        data.x = deg.view(-1, 1)
        return data


def num_graphs(data):
    if data.batch is not None:
        return data.num_graphs
    else:
        return data.x.size(0)


def train(model, optimizer, loader,a_loss, a_storage=1):
    model.train()

    total_loss = 0
    for data in loader:
         if data.num_nodes>650:
            optimizer.zero_grad()
            data = data.to(device)
            out,bit_sum = model(data)
            loss = F.cross_entropy(out, data.y.view(-1))
            loss_store = a_loss*F.relu(bit_sum-a_storage)**2
            loss_store.backward(retain_graph=True)
            loss.backward()
            total_loss += loss.item() * num_graphs(data)
            optimizer.step()
    return total_loss / len(loader.dataset)


def eval_acc(model, loader):
    model.eval()
    correct = 0
    for data in loader:        
        if data.num_nodes>650:
            data = data.to(device)
            with torch.no_grad():
                pred = model(data)[0].max(1)[1]                   
        correct+=pred.eq(data.y.view(-1)).sum().item()
    return correct / len(loader.dataset)




            

def eval_loss(model, loader):
    model.eval()

    loss = 0
    for data in loader:
        if data.num_nodes>650:
            data = data.to(device)
            with torch.no_grad():
                out = model(data)[0]
            loss += F.cross_entropy(out, data.y.view(-1), reduction="sum").item()
    return loss / len(loader.dataset)


# Real k_fold
def k_fold(dataset, folds):
    skf = StratifiedKFold(folds, shuffle=True, random_state=12345)

    test_indices, train_indices = [], []
    for _, idx in skf.split(torch.zeros(len(dataset)), dataset.data.y):
        test_indices.append(torch.from_numpy(idx))

    val_indices = [test_indices[i - 1] for i in range(folds)]

    for i in range(folds):
        train_mask = torch.ones(len(dataset), dtype=torch.bool)
        train_mask[test_indices[i]] = 0
        train_mask[val_indices[i]] = 0
        train_indices.append(train_mask.nonzero().view(-1))

    return train_indices, test_indices, val_indices


def load_checkpoint(model, checkpoint):
    if checkpoint != 'No':
        print("loading checkpoint...")
        model_dict = model.state_dict()
        modelCheckpoint = torch.load(checkpoint)
        pretrained_dict = modelCheckpoint['state_dict']
        new_dict = {k: v for k, v in pretrained_dict.items() if ((k in model_dict.keys()))}
        model_dict.update(new_dict)
        print('Total : {}, update: {}'.format(len(pretrained_dict), len(new_dict)))
        model.load_state_dict(model_dict)
        print("loaded finished!")
    return model

## Definition of Requirment Parameters as args

In [6]:
import sys
import argparse

# Clearing the arguments
sys.argv = ['']


parser = argparse.ArgumentParser()
parser.add_argument('--model',type=str,default='GIN')
parser.add_argument('--gpu_id',type=int,default=0)
parser.add_argument('--dataset_name',type=str,default='BBBP')
parser.add_argument('--num_deg',type=int,default=1000)
parser.add_argument('--num_layers', type=int, default=5)
parser.add_argument('--hidden_units',type=int,default=64)
parser.add_argument('--batch-size',type=int,default=64)
parser.add_argument('--bit',type=int,default=4)
parser.add_argument('--max_epoch',type=int,default=100)
parser.add_argument('--max_cycle',type=int,default=2000)
parser.add_argument('--folds',type=int,default=10)
parser.add_argument('--weight_decay',type=float,default=0)
parser.add_argument('--lr',type=float,default=0.01)
parser.add_argument('--a_loss',type=float,default=0.001)
parser.add_argument('--lr_quant_scale_fea',type=float,default=0.02)
parser.add_argument('--lr_quant_scale_xw',type=float,default=1e-2)
parser.add_argument('--lr_quant_scale_weight',type=float,default=0.02)
parser.add_argument('--lr_quant_bit_fea',type=float,default=0.008)
parser.add_argument('--lr_quant_bit_weight',type=float,default=0.0001)
parser.add_argument('--lr_step_size',type=int, default=50)
parser.add_argument('--lr_decay_factor',type=float,default=0.5)
parser.add_argument('--lr_schedule_patience',type=int,default=10)
parser.add_argument('--is_naive',type=bool,default=False)
###############################################################
parser.add_argument('--resume',type=bool,default=True)
parser.add_argument('--store_ckpt',type=bool,default=True)
parser.add_argument('--uniform',type=bool,default=True)
parser.add_argument('--use_norm_quant',type=bool,default=True)
###############################################################
# The target memory size of nodes features
parser.add_argument('--a_storage',type=float,default=1)
# Path to results
parser.add_argument('--result_folder',type=str,default='result')
# Path to checkpoint
parser.add_argument('--check_folder',type=str,default='checkpoint')
# Path to dataset
parser.add_argument('--path2dataset',type=str,default='/')

args = parser.parse_args()
print(args)

Namespace(model='GIN', gpu_id=0, dataset_name='BBBP', num_deg=1000, num_layers=5, hidden_units=64, batch_size=64, bit=4, max_epoch=100, max_cycle=2000, folds=10, weight_decay=0, lr=0.01, a_loss=0.001, lr_quant_scale_fea=0.02, lr_quant_scale_xw=0.01, lr_quant_scale_weight=0.02, lr_quant_bit_fea=0.008, lr_quant_bit_weight=0.0001, lr_step_size=50, lr_decay_factor=0.5, lr_schedule_patience=10, is_naive=False, resume=True, store_ckpt=True, uniform=True, use_norm_quant=True, a_storage=1, result_folder='result', check_folder='checkpoint', path2dataset='/')


In [7]:
###############################################################
model = args.model
dataset_name = args.dataset_name
num_layers = args.num_layers
hidden_units=args.hidden_units
bit=args.bit
max_epoch = args.max_epoch
resume = args.resume


# Path direction
path2result = args.result_folder+'/'+args.model+'_'+dataset_name
path2check = args.check_folder+'/'+args.model+'_'+dataset_name
if not os.path.exists(path2result):
    os.makedirs(path2result)
if not os.path.exists(path2check):
    os.makedirs(path2check)
###############################################################


## Loading Dataset and Normalization

In [8]:
import os
import glob
import json
import torch
import pickle
import numpy as np
import os.path as osp
from torch_geometric.datasets import MoleculeNet
from torch_geometric.utils import dense_to_sparse
from torch.utils.data import random_split, Subset
from torch_geometric.data import Data, InMemoryDataset, DataLoader


def undirected_graph(data):
    data.edge_index = torch.cat([torch.stack([data.edge_index[1], data.edge_index[0]], dim=0),
                                 data.edge_index], dim=1)
    return data


def split(data, batch):
    # i-th contains elements from slice[i] to slice[i+1]
    node_slice = torch.cumsum(torch.from_numpy(np.bincount(batch)), 0)
    node_slice = torch.cat([torch.tensor([0]), node_slice])
    row, _ = data.edge_index
    edge_slice = torch.cumsum(torch.from_numpy(np.bincount(batch[row])), 0)
    edge_slice = torch.cat([torch.tensor([0]), edge_slice])

    # Edge indices should start at zero for every graph.
    data.edge_index -= node_slice[batch[row]].unsqueeze(0)
    data.__num_nodes__ = np.bincount(batch).tolist()

    slices = dict()
    slices['x'] = node_slice
    slices['edge_index'] = edge_slice
    slices['y'] = torch.arange(0, batch[-1] + 2, dtype=torch.long)
    return data, slices


def read_file(folder, prefix, name):
    file_path = osp.join(folder, prefix + f'_{name}.txt')
    return np.genfromtxt(file_path, dtype=np.int64)


def read_sentigraph_data(folder: str, prefix: str):
    txt_files = glob.glob(os.path.join(folder, "{}_*.txt".format(prefix)))
    json_files = glob.glob(os.path.join(folder, "{}_*.json".format(prefix)))
    txt_names = [f.split(os.sep)[-1][len(prefix) + 1:-4] for f in txt_files]
    json_names = [f.split(os.sep)[-1][len(prefix) + 1:-5] for f in json_files]
    names = txt_names + json_names

    with open(os.path.join(folder, prefix+"_node_features.pkl"), 'rb') as f:
        x: np.array = pickle.load(f)
    x: torch.FloatTensor = torch.from_numpy(x)
    edge_index: np.array = read_file(folder, prefix, 'edge_index')
    edge_index: torch.tensor = torch.tensor(edge_index, dtype=torch.long).T
    batch: np.array = read_file(folder, prefix, 'node_indicator') - 1     # from zero
    y: np.array = read_file(folder, prefix, 'graph_labels')
    y: torch.tensor = torch.tensor(y, dtype=torch.long)

    supplement = dict()
    if 'split_indices' in names:
        split_indices: np.array = read_file(folder, prefix, 'split_indices')
        split_indices = torch.tensor(split_indices, dtype=torch.long)
        supplement['split_indices'] = split_indices
    if 'sentence_tokens' in names:
        with open(os.path.join(folder, prefix + '_sentence_tokens.json')) as f:
            sentence_tokens: dict = json.load(f)
        supplement['sentence_tokens'] = sentence_tokens

    data = Data(x=x, edge_index=edge_index, y=y)
    data, slices = split(data, batch)

    return data, slices, supplement


def read_syn_data(folder: str, prefix):
    with open(os.path.join(folder, f"{prefix}.pkl"), 'rb') as f:
        adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, edge_label_matrix = pickle.load(f)

    x = torch.from_numpy(features).float()
    y = train_mask.reshape(-1, 1) * y_train + val_mask.reshape(-1, 1) * y_val + test_mask.reshape(-1, 1) * y_test
    y = torch.from_numpy(np.where(y)[1])
    edge_index = dense_to_sparse(torch.from_numpy(adj))[0]
    data = Data(x=x, y=y, edge_index=edge_index)
    data.train_mask = torch.from_numpy(train_mask)
    data.val_mask = torch.from_numpy(val_mask)
    data.test_mask = torch.from_numpy(test_mask)
    return data


def read_ba2motif_data(folder: str, prefix):
    with open(os.path.join(folder, f"{prefix}.pkl"), 'rb') as f:
        dense_edges, node_features, graph_labels = pickle.load(f)

    data_list = []
    for graph_idx in range(dense_edges.shape[0]):
        data_list.append(Data(x=torch.from_numpy(node_features[graph_idx]).float(),
                              edge_index=dense_to_sparse(torch.from_numpy(dense_edges[graph_idx]))[0],
                              y=torch.from_numpy(np.where(graph_labels[graph_idx])[0])))
    return data_list


def get_dataset(dataset_dir, dataset_name, task=None):
    sync_dataset_dict = {
        'BA_2Motifs'.lower(): 'BA_2Motifs',
        'BA_Shapes'.lower(): 'BA_shapes',
        'BA_Community'.lower(): 'BA_Community',
        'Tree_Cycle'.lower(): 'Tree_Cycle',
        'Tree_Grids'.lower(): 'Tree_Grids',
    }
    sentigraph_names = ['Graph_SST2', 'Graph_Twitter', 'Graph_SST5']
    sentigraph_names = [name.lower() for name in sentigraph_names]
    molecule_net_dataset_names = [name.lower() for name in MoleculeNet.names.keys()]

    if dataset_name.lower() == 'MUTAG'.lower():
        return load_MUTAG(dataset_dir, 'MUTAG')
    elif dataset_name.lower() in sync_dataset_dict.keys():
        sync_dataset_filename = sync_dataset_dict[dataset_name.lower()]
        return load_syn_data(dataset_dir, sync_dataset_filename)
    elif dataset_name.lower() in molecule_net_dataset_names:
        return load_MolecueNet(dataset_dir, dataset_name, task)
    elif dataset_name.lower() in sentigraph_names:
        return load_SeniGraph(dataset_dir, dataset_name)
    else:
        raise NotImplementedError


class MUTAGDataset(InMemoryDataset):
    def __init__(self, root, name, transform=None, pre_transform=None):
        self.root = root
        self.name = name.upper()
        super(MUTAGDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    def __len__(self):
        return len(self.slices['x']) - 1

    @property
    def raw_dir(self):
        return os.path.join(self.root, self.name, 'raw')

    @property
    def raw_file_names(self):
        return ['MUTAG_A', 'MUTAG_graph_labels', 'MUTAG_graph_indicator', 'MUTAG_node_labels', 'MUTAG_edge_labels']

    @property
    def processed_dir(self):
        return os.path.join(self.root, self.name, 'processed')

    @property
    def processed_file_names(self):
        return ['data.pt']

    def process(self):
        r"""Processes the dataset to the :obj:`self.processed_dir` folder."""
        with open(os.path.join(self.raw_dir, 'MUTAG_node_labels.txt'), 'r') as f:
            nodes_all_temp = f.read().splitlines()
            nodes_all = [int(i) for i in nodes_all_temp]

        adj_all = np.zeros((len(nodes_all), len(nodes_all)))
        with open(os.path.join(self.raw_dir, 'MUTAG_A.txt'), 'r') as f:
            adj_list = f.read().splitlines()
        for item in adj_list:
            lr = item.split(', ')
            l = int(lr[0])
            r = int(lr[1])
            adj_all[l - 1, r - 1] = 1

        with open(os.path.join(self.raw_dir, 'MUTAG_graph_indicator.txt'), 'r') as f:
            graph_indicator_temp = f.read().splitlines()
            graph_indicator = [int(i) for i in graph_indicator_temp]
            graph_indicator = np.array(graph_indicator)

        with open(os.path.join(self.raw_dir, 'MUTAG_graph_labels.txt'), 'r') as f:
            graph_labels_temp = f.read().splitlines()
            graph_labels = [int(i) for i in graph_labels_temp]

        data_list = []
        for i in range(1, 189):
            idx = np.where(graph_indicator == i)
            graph_len = len(idx[0])
            adj = adj_all[idx[0][0]:idx[0][0] + graph_len, idx[0][0]:idx[0][0] + graph_len]
            label = int(graph_labels[i - 1] == 1)
            feature = nodes_all[idx[0][0]:idx[0][0] + graph_len]
            nb_clss = 7
            targets = np.array(feature).reshape(-1)
            one_hot_feature = np.eye(nb_clss)[targets]
            data_example = Data(x=torch.from_numpy(one_hot_feature).float(),
                                edge_index=dense_to_sparse(torch.from_numpy(adj))[0],
                                y=label)
            data_list.append(data_example)

        torch.save(self.collate(data_list), self.processed_paths[0])


class SentiGraphDataset(InMemoryDataset):
    def __init__(self, root, name, transform=None, pre_transform=undirected_graph):
        self.name = name
        super(SentiGraphDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices, self.supplement = torch.load(self.processed_paths[0])

    @property
    def raw_dir(self):
        return osp.join(self.root, self.name, 'raw')

    @property
    def processed_dir(self):
        return osp.join(self.root, self.name, 'processed')

    @property
    def raw_file_names(self):
        return ['node_features', 'node_indicator', 'sentence_tokens', 'edge_index',
                'graph_labels', 'split_indices']

    @property
    def processed_file_names(self):
        return ['data.pt']

    def process(self):
        # Read data into huge `Data` list.
        self.data, self.slices, self.supplement \
              = read_sentigraph_data(self.raw_dir, self.name)

        if self.pre_filter is not None:
            data_list = [self.get(idx) for idx in range(len(self))]
            data_list = [data for data in data_list if self.pre_filter(data)]
            self.data, self.slices = self.collate(data_list)

        if self.pre_transform is not None:
            data_list = [self.get(idx) for idx in range(len(self))]
            data_list = [self.pre_transform(data) for data in data_list]
            self.data, self.slices = self.collate(data_list)
        torch.save((self.data, self.slices, self.supplement), self.processed_paths[0])


class SynGraphDataset(InMemoryDataset):
    def __init__(self, root, name, transform=None, pre_transform=None):
        self.name = name
        super(SynGraphDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_dir(self):
        return osp.join(self.root, self.name, 'raw')

    @property
    def processed_dir(self):
        return osp.join(self.root, self.name, 'processed')

    @property
    def raw_file_names(self):
        return [f"{self.name}.pkl"]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def process(self):
        # Read data into huge `Data` list.
        data = read_syn_data(self.raw_dir, self.name)
        data = data if self.pre_transform is None else self.pre_transform(data)
        torch.save(self.collate([data]), self.processed_paths[0])


class BA2MotifDataset(InMemoryDataset):
    def __init__(self, root, name, transform=None, pre_transform=None):
        self.name = name
        super(BA2MotifDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_dir(self):
        return osp.join(self.root, self.name, 'raw')

    @property
    def processed_dir(self):
        return osp.join(self.root, self.name, 'processed')

    @property
    def raw_file_names(self):
        return [f"{self.name}.pkl"]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def process(self):
        # Read data into huge `Data` list.
        data_list = read_ba2motif_data(self.raw_dir, self.name)

        if self.pre_filter is not None:
            data_list = [self.get(idx) for idx in range(len(self))]
            data_list = [data for data in data_list if self.pre_filter(data)]
            self.data, self.slices = self.collate(data_list)

        if self.pre_transform is not None:
            data_list = [self.get(idx) for idx in range(len(self))]
            data_list = [self.pre_transform(data) for data in data_list]
            self.data, self.slices = self.collate(data_list)

        torch.save(self.collate(data_list), self.processed_paths[0])


def load_MUTAG(dataset_dir, dataset_name):
    """ 188 molecules where label = 1 denotes mutagenic effect """
    dataset = MUTAGDataset(root=dataset_dir, name=dataset_name)
    return dataset


def load_syn_data(dataset_dir, dataset_name):
    """ The synthetic dataset """
    if dataset_name.lower() == 'BA_2Motifs'.lower():
        dataset = BA2MotifDataset(root=dataset_dir, name=dataset_name)
    else:
        dataset = SynGraphDataset(root=dataset_dir, name=dataset_name)
    dataset.node_type_dict = {k: v for k, v in enumerate(range(dataset.num_classes))}
    dataset.node_color = None
    return dataset


def load_MolecueNet(dataset_dir, dataset_name, task=None):
    """ Attention the multi-task problems not solved yet """
    molecule_net_dataset_names = {name.lower(): name for name in MoleculeNet.names.keys()}
    dataset = MoleculeNet(root=dataset_dir, name=molecule_net_dataset_names[dataset_name.lower()])
    dataset.data.x = dataset.data.x.float()
    if task is None:
        dataset.data.y = dataset.data.y.squeeze().long()
    else:
        dataset.data.y = dataset.data.y[task].long()
    dataset.node_type_dict = None
    dataset.node_color = None
    return dataset


def load_SeniGraph(dataset_dir, dataset_name):
    dataset = SentiGraphDataset(root=dataset_dir, name=dataset_name)
    return dataset


def get_dataloader(dataset, batch_size, random_split_flag=True, data_split_ratio=None, seed=2):
    """
    Args:
        dataset:
        batch_size: int
        random_split_flag: bool
        data_split_ratio: list, training, validation and testing ratio
        seed: random seed to split the dataset randomly
    Returns:
        a dictionary of training, validation, and testing dataLoader
    """

    if not random_split_flag and hasattr(dataset, 'supplement'):
        assert 'split_indices' in dataset.supplement.keys(), "split idx"
        split_indices = dataset.supplement['split_indices']
        train_indices = torch.where(split_indices == 0)[0].numpy().tolist()
        dev_indices = torch.where(split_indices == 1)[0].numpy().tolist()
        test_indices = torch.where(split_indices == 2)[0].numpy().tolist()

        train = Subset(dataset, train_indices)
        eval = Subset(dataset, dev_indices)
        test = Subset(dataset, test_indices)
    else:
        num_train = int(0.8 * len(dataset))
        num_eval = int(0.1 * len(dataset))
        num_test = len(dataset) - num_train - num_eval

        train, eval, test = random_split(dataset, lengths=[num_train, num_eval, num_test],
                                         generator=torch.Generator().manual_seed(seed))

    dataloader = dict()
    dataloader['train'] = DataLoader(train, batch_size=batch_size, shuffle=True)
    dataloader['eval'] = DataLoader(eval, batch_size=batch_size, shuffle=False)
    dataloader['test'] = DataLoader(test, batch_size=batch_size, shuffle=False)
    return dataloader


### Loading dataset

In [9]:
if(args.resume==True):
    file_name = path2result+'/'+args.model+'_'+str(hidden_units)+'_'+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.txt'
    if not os.path.exists(file_name):
            with open(file_name, 'w') as f:
                for key, value in vars(args).items():
                    f.write('%s:%s\n'%(key, value))
if(args.dataset_name=='BBBP'):
    dataset = get_dataset(args.path2dataset, args.dataset_name)


C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [10]:

if dataset.data.x is None:
    max_degree = 0
    degs = []
    for data in dataset:
        degs += [degree(data.edge_index[0], dtype=torch.long)]
        max_degree = max(max_degree, degs[-1].max().item())

    if max_degree < 1000:
        dataset.transform = T.OneHotDegree(max_degree)
    else:
        deg = torch.cat(degs, dim=0).to(torch.float)
        mean, std = deg.mean().item(), deg.std().item()
        dataset.transform = NormalizedDegree(mean, std)

## Training Process

In [11]:

# writer = SummaryWriter(log_dir=path2log)
args.batch_size=16
args.max_epoch=100
args.batch_size=32
max_acc = 0.79
for i in range(1):
    accu = []
    accu_max = []
    # model=qGIN(dataset, args.num_layers,hidden_units=args.hidden_units,bit=args.bit, quant_method=args.quant_method).to(device)
    # model = GIN(2,32).to(device)
    # setup_seed(12345)

    for fold, (train_idx, test_idx, val_idx) in enumerate(zip(*k_fold(dataset, args.folds))):
        print_max_acc=0
        train_dataset = dataset[train_idx.tolist()]
        test_dataset = dataset[test_idx.tolist()]
        val_dataset = dataset[val_idx.tolist()]
        train_loader = DataLoader(train_dataset, args.batch_size, num_workers=0,shuffle=False, drop_last=True)
        val_loader = DataLoader(val_dataset, args.batch_size,num_workers=0,shuffle=False,drop_last=True)
        test_loader = DataLoader(test_dataset, args.batch_size,num_workers=0,shuffle=False, drop_last=True)
        k=0
       
     
        model=qGIN(train_dataset, args.num_layers,hidden_units=args.hidden_units,bit=args.bit, is_q=True,
                num_deg=args.num_deg,
                uniform=args.uniform).to(device)
        weight_paras,quant_paras_bit_weight, quant_paras_bit_fea, quant_paras_scale_weight, quant_paras_scale_fea, quant_paras_scale_xw, quant_paras_bit_xw, other_paras = paras_group(model)
        # quant_paras_bit.requires_grad = False
        optimizer = torch.optim.Adam([{'params':weight_paras},
                                    {'params':quant_paras_scale_weight,'lr':args.lr_quant_scale_weight,'weight_decay':0},
                                    {'params':quant_paras_scale_fea,'lr':args.lr_quant_scale_fea,'weight_decay':0},
                                    {'params':quant_paras_scale_xw,'lr':args.lr_quant_scale_xw,'weight_decay':0},
                                    # {'params':quant_paras_bit_weight,'lr':args.lr_quant_bit_weight,'weight_decay':0},
                                    {'params':quant_paras_bit_fea,'lr':args.lr_quant_bit_fea,'weight_decay':0},
                                    {'params':other_paras}],
                                    lr=args.lr, weight_decay=args.weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_step_size, gamma=args.lr_decay_factor)

        # if (os.path.exists(path2check)):
        #     model = load_checkpoint(model,path2check)

        for epoch in range(5):
            t = tqdm(epoch)
            train_loss=0
            train_loss = train(model,optimizer,train_loader,args.a_loss, args.a_storage)
            start = time.process_time()
            val_loss = eval_loss(model,val_loader)
            end = time.process_time()
            #acc = eval_acc(model,test_loader)
            correct = 0
            for data in test_loader:

                data = data.to(device)
                if data.num_nodes>650:
                        with torch.no_grad():
                            pred = model(data)[0].max(1)[1]
                   
                correct += pred.eq(data.y.view(-1)).sum().item()
            acc=correct / len(test_loader.dataset)
            
            
            
            
            # for name,param in model.named_parameters():
            #     a=param.grad
            #     if(a!=None):
            #         writer.add_histogram(tag=name+'_grad', values=a, global_step=epoch)
            scheduler.step()
            t.set_postfix(
                        {
                            "Train_Loss": "{:05.3f}".format(train_loss),
                            "Val_Loss": "{:05.3f}".format(val_loss),
                            "Acc": "{:05.3f}".format(acc),
                            "Epoch":"{:05.1f}".format(epoch),
                            "Fold":"{:05.1f}".format(fold),
                        }
                    )
            accu.append(acc)
            if(acc>max_acc):
                path = path2check+'/'+args.model+'_'+str(hidden_units)+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.pth.tar'
                max_acc = acc
                torch.save({'state_dict': model.state_dict(), 'best_accu': acc,}, path)
            if(acc>print_max_acc):
                print_max_acc = acc
            if(args.resume==True):
                f = open(file_name,'a')
                f.write(str(acc))
                f.write('\n')
        if(args.resume==True):
            f = open(file_name,'a')
            f.write('The max accu of the {} runs is:'.format(i))
            f.write(str(print_max_acc))
            f.write('\n')
    # pdb.set_trace()
    accu_new = torch.tensor(accu)
    #accu = accu_new.view(args.folds,args.max_epoch)
    accu = accu_new.view(args.folds,5)
    _, argmax = accu.max(dim=1)
    accu = accu[torch.arange(args.folds, dtype=torch.long), argmax]
    acc_mean = accu.mean().item()
    acc_std = accu.std().item()
    desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
    print('desc:', desc  )  
    if(args.resume==True):
        f = open(file_name,'a')
        f.write('The result is:')
        f.write(desc)
        f.write('\n')
print("finish")

 

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
0it [00:16, ?it/s, Train_Loss=0.492, Val_Loss=2.519, Acc=0.702, Epoch=000.0, Fold=000.0]
0it [00:16, ?it/s, Train_Loss=0.492, Val_Loss=2.519, Acc=0.702, Epoch=000.0, Fold=000.0]

0it [00:17, ?it/s, Train_Loss=0.738, Val_Loss=0.977, Acc=0.702, Epoch=001.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.546, Val_Loss=0.658, Acc=0.702, Epoch=002.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.546, Val_Loss=0.658, Acc=0.702, Epoch=002.0, Fold=000.0]

0it [00:14, ?it/s, Train_Loss=0.542, Val_Loss=1.083, Acc=0.702, Epoch=003.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.524, Val_Loss=0.548, Acc=0.702, Epoch=004.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.524, Val_Loss=0.548, Acc=0.702, Epoch=004.0, Fold=000.0]

0it [00:17, ?it/s, Train_Loss=0.452, Val_Loss=2.930, Acc=0.702, Epoch=000.0, Fo

desc: 0.701 ± 0.002
finish


## Manual Measurement

In [11]:
Eva_final=dict()



AWQ_model_accuracy=[]
T_AWQ_model=[]
Num_parm_AWQ_model=[]
AWQ_model_size=[]

In [16]:

# writer = SummaryWriter(log_dir=path2log)
args.batch_size=16
args.max_epoch=100
args.batch_size=32
max_acc = 0.79
for i in range(3):
    Eva=dict()
    accu = []
    accu_max = []
    # model=qGIN(dataset, args.num_layers,hidden_units=args.hidden_units,bit=args.bit, quant_method=args.quant_method).to(device)
    # model = GIN(2,32).to(device)
    # setup_seed(12345)

    for fold, (train_idx, test_idx, val_idx) in enumerate(zip(*k_fold(dataset, args.folds))):
        print_max_acc=0
        train_dataset = dataset[train_idx.tolist()]
        test_dataset = dataset[test_idx.tolist()]
        val_dataset = dataset[val_idx.tolist()]
        train_loader = DataLoader(train_dataset, args.batch_size, num_workers=0,shuffle=False, drop_last=True)
        val_loader = DataLoader(val_dataset, args.batch_size,num_workers=0,shuffle=False,drop_last=True)
        test_loader = DataLoader(test_dataset, args.batch_size,num_workers=0,shuffle=False, drop_last=True)
        k=0
       
     
        model=qGIN(train_dataset, args.num_layers,hidden_units=args.hidden_units,bit=args.bit, is_q=True,
                num_deg=args.num_deg,
                uniform=args.uniform).to(device)
        weight_paras,quant_paras_bit_weight, quant_paras_bit_fea, quant_paras_scale_weight, quant_paras_scale_fea, quant_paras_scale_xw, quant_paras_bit_xw, other_paras = paras_group(model)
        # quant_paras_bit.requires_grad = False
        optimizer = torch.optim.Adam([{'params':weight_paras},
                                    {'params':quant_paras_scale_weight,'lr':args.lr_quant_scale_weight,'weight_decay':0},
                                    {'params':quant_paras_scale_fea,'lr':args.lr_quant_scale_fea,'weight_decay':0},
                                    {'params':quant_paras_scale_xw,'lr':args.lr_quant_scale_xw,'weight_decay':0},
                                    # {'params':quant_paras_bit_weight,'lr':args.lr_quant_bit_weight,'weight_decay':0},
                                    {'params':quant_paras_bit_fea,'lr':args.lr_quant_bit_fea,'weight_decay':0},
                                    {'params':other_paras}],
                                    lr=args.lr, weight_decay=args.weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_step_size, gamma=args.lr_decay_factor)

        # if (os.path.exists(path2check)):
        #     model = load_checkpoint(model,path2check)

        for epoch in range(5):
            t = tqdm(epoch)
            train_loss=0
            train_loss = train(model,optimizer,train_loader,args.a_loss, args.a_storage)
            start = time.process_time()
            val_loss = eval_loss(model,val_loader)
            end = time.process_time()
            #acc = eval_acc(model,test_loader)
            correct = 0
            for data in test_loader:
                    data = data.to(device)
                    if data.num_nodes>650:
                            with torch.no_grad():
                                pred = model(data)[0].max(1)[1]

                    correct += pred.eq(data.y.view(-1)).sum().item()
            acc=correct / len(test_loader.dataset)
            
    
            # for name,param in model.named_parameters():
            #     a=param.grad
            #     if(a!=None):
            #         writer.add_histogram(tag=name+'_grad', values=a, global_step=epoch)
            scheduler.step()
            t.set_postfix(
                        {
                            "Train_Loss": "{:05.3f}".format(train_loss),
                            "Val_Loss": "{:05.3f}".format(val_loss),
                            "Acc": "{:05.3f}".format(acc),
                            "Epoch":"{:05.1f}".format(epoch),
                            "Fold":"{:05.1f}".format(fold),
                        }
                    )
            accu.append(acc)
            if(acc>max_acc):
                path = path2check+'/'+args.model+'_'+str(hidden_units)+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.pth.tar'
                max_acc = acc
                torch.save({'state_dict': model.state_dict(), 'best_accu': acc,}, path)
            if(acc>print_max_acc):
                print_max_acc = acc
            if(args.resume==True):
                f = open(file_name,'a')
                f.write(str(acc))
                f.write('\n')
        if(args.resume==True):
            f = open(file_name,'a')
            f.write('The max accu of the {} runs is:'.format(i))
            f.write(str(print_max_acc))
            f.write('\n')
    # pdb.set_trace()
    accu_new = torch.tensor(accu)
    #accu = accu_new.view(args.folds,args.max_epoch)
    accu = accu_new.view(args.folds,5)
    _, argmax = accu.max(dim=1)
    accu = accu[torch.arange(args.folds, dtype=torch.long), argmax]
    acc_mean = accu.mean().item()
    acc_std = accu.std().item()
    desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
    print('desc:', desc  )  
    
    
    t0=time.time()
    awq_model_accuracy=eval_acc(model,test_loader)
    t1=time.time()
    t_awq_model=t1 - t0
    ###
    awq_model_size = get_model_size(model,count_nonzero_only=True)
    num_parm_awq_model=get_num_parameters(model, count_nonzero_only=True)
    ###
    #base_model_size=755424
    print(f"awq model has accuracy on test set={awq_model_accuracy:.2f}%")
    print(f"awq model has size={awq_model_size/MiB:.2f} MiB")
    print(f"The time inference of awq model is ={t_awq_model}") 
    print(f"The number of parametrs of awq model is:{num_parm_awq_model}")
    #print(f"awq model has size={awq_model_size/MiB:.2f} MiB, "
          #f"which is {base_model_size/awq_model_size:.2f} X smaller than "
          #f"the {base_model_size/MiB:.2f} MiB base model")
    
    #Update my Eva dictionary
    Eva.update({'awq model accuracy': awq_model_accuracy,
                'time inference of awq model': t_awq_model,
                'number parmameters of awq model': num_parm_awq_model,
                'size of awq model': awq_model_size})  
    
    AWQ_model_accuracy.append(Eva['awq model accuracy'])
    T_AWQ_model.append(Eva['time inference of awq model'])
    Num_parm_AWQ_model.append(int(Eva['number parmameters of awq model']))
    AWQ_model_size.append(int(Eva['size of awq model']))

    
 

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
0it [32:49, ?it/s, Train_Loss=0.524, Val_Loss=0.563, Acc=0.698, Epoch=004.0, Fold=009.0]
0it [00:15, ?it/s, Train_Loss=0.422, Val_Loss=3.663, Acc=0.702, Epoch=000.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.422, Val_Loss=3.663, Acc=0.702, Epoch=000.0, Fold=000.0]

0it [00:13, ?it/s, Train_Loss=

desc: 0.701 ± 0.002


C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


awq model has accuracy on test set=0.70%
awq model has size=0.26 MiB
The time inference of awq model is =0.9862549304962158
The number of parametrs of awq model is:66939


0it [00:16, ?it/s, Train_Loss=0.583, Val_Loss=0.514, Acc=0.698, Epoch=004.0, Fold=009.0]
0it [00:15, ?it/s, Train_Loss=0.448, Val_Loss=2.887, Acc=0.702, Epoch=000.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.448, Val_Loss=2.887, Acc=0.702, Epoch=000.0, Fold=000.0]

0it [00:14, ?it/s, Train_Loss=0.704, Val_Loss=0.881, Acc=0.702, Epoch=001.0, Fold=000.0]
0it [00:14, ?it/s, Train_Loss=0.706, Val_Loss=0.531, Acc=0.702, Epoch=002.0, Fold=000.0]
0it [00:14, ?it/s, Train_Loss=0.706, Val_Loss=0.531, Acc=0.702, Epoch=002.0, Fold=000.0]

0it [00:14, ?it/s, Train_Loss=0.535, Val_Loss=0.524, Acc=0.702, Epoch=003.0, Fold=000.0]
0it [00:14, ?it/s, Train_Loss=0.461, Val_Loss=0.823, Acc=0.702, Epoch=004.0, Fold=000.0]
0it [00:14, ?it/s, Train_Loss=0.461, Val_Loss=0.823, Acc=0.702, Epoch=004.0, Fold=000.0]

0it [00:15, ?it/s, Train_Loss=0.458, Val_Loss=5.303, Acc=0.702, Epoch=000.0, Fold=001.0]
0it [00:14, ?it/s, Train_Loss=0.772, Val_Loss=0.909, Acc=0.702, Epoch=001.0, Fold=001.0]
0it [00:14, ?it/s,

desc: 0.701 ± 0.002


C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\deprecation.py:22: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


awq model has accuracy on test set=0.70%
awq model has size=0.26 MiB
The time inference of awq model is =1.0261666774749756
The number of parametrs of awq model is:66939


0it [00:17, ?it/s, Train_Loss=0.536, Val_Loss=0.565, Acc=0.698, Epoch=004.0, Fold=009.0]
0it [00:15, ?it/s, Train_Loss=0.409, Val_Loss=3.073, Acc=0.702, Epoch=000.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.409, Val_Loss=3.073, Acc=0.702, Epoch=000.0, Fold=000.0]

0it [00:15, ?it/s, Train_Loss=0.727, Val_Loss=0.908, Acc=0.702, Epoch=001.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.693, Val_Loss=0.694, Acc=0.702, Epoch=002.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.693, Val_Loss=0.694, Acc=0.702, Epoch=002.0, Fold=000.0]

0it [00:15, ?it/s, Train_Loss=0.535, Val_Loss=0.535, Acc=0.702, Epoch=003.0, Fold=000.0]
0it [00:14, ?it/s, Train_Loss=0.533, Val_Loss=0.527, Acc=0.702, Epoch=004.0, Fold=000.0]
0it [00:15, ?it/s, Train_Loss=0.533, Val_Loss=0.527, Acc=0.702, Epoch=004.0, Fold=000.0]

0it [00:16, ?it/s, Train_Loss=0.467, Val_Loss=3.650, Acc=0.702, Epoch=000.0, Fold=001.0]
0it [00:15, ?it/s, Train_Loss=0.770, Val_Loss=1.832, Acc=0.702, Epoch=001.0, Fold=001.0]
0it [00:15, ?it/s,

desc: 0.701 ± 0.002
awq model has accuracy on test set=0.70%
awq model has size=0.26 MiB
The time inference of awq model is =0.9805107116699219
The number of parametrs of awq model is:66939


In [20]:


AWQ_model_accuracy_mean =stat.mean(AWQ_model_accuracy)
AWQ_model_accuracy_std = stat.stdev(AWQ_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ model accuracy':float(format(AWQ_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of AWQ model accuracy':float(format(AWQ_model_accuracy_std, '.3f'))})
                 

t_AWQ_model_mean = stat.mean(T_AWQ_model)
t_AWQ_model_std =stat.stdev(T_AWQ_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of AWQ model':float(format(t_AWQ_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of AWQ model':float(format(t_AWQ_model_std, '.3f'))})

num_parm_AWQ_model_mean = stat.mean(Num_parm_AWQ_model)
num_parm_AWQ_model_std = stat.stdev(Num_parm_AWQ_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of AWQ model':num_parm_AWQ_model_mean})
Eva_final.update({'Std of number parmameters of AWQ model':num_parm_AWQ_model_std})

AWQ_model_size_mean =stat.mean( AWQ_model_size)
AWQ_model_size_std = stat.stdev(AWQ_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'AWQ model size':AWQ_model_size_mean})
Eva_final.update({'Std of AWQ_model_size':AWQ_model_size_std })


#################################



print(f"All measurement about Quatization process is:  ")   
Eva_final

All measurement about Quatization process is:  


{'AWQ model accuracy': 0.698,
 'Std of AWQ model accuracy': 0.0,
 'time inference of AWQ model': 0.998,
 'Std of time inference of AWQ model': 0.025,
 'number parmameters of AWQ model': 66939,
 'Std of number parmameters of AWQ model': 0.0,
 'AWQ model size': 2142048,
 'Std of AWQ_model_size': 0.0}

In [ ]:
#All measurement about Quatization process is:  
{'AWQ model accuracy': 0.698,
 'Std of AWQ model accuracy': 0.0,
 'time inference of AWQ model': 0.998,
 'Std of time inference of AWQ model': 0.025,
 'number parmameters of AWQ model': 66939,
 'Std of number parmameters of AWQ model': 0.0,
 'AWQ model size': 2142048,
 'Std of AWQ_model_size': 0.0}

In [21]:
AWQ_model_accuracy.append(Eva_final['AWQ model accuracy'] )
AWQ_model_accuracy.append(Eva_final['Std of AWQ model accuracy'] )
T_AWQ_model.append(Eva_final['time inference of AWQ model'])
T_AWQ_model.append(Eva_final['Std of time inference of AWQ model'])
Num_parm_AWQ_model.append(Eva_final['number parmameters of AWQ model'] )
Num_parm_AWQ_model.append(Eva_final['Std of number parmameters of AWQ model'])
AWQ_model_size.append(Eva_final['AWQ model size'] )
AWQ_model_size.append(Eva_final['Std of AWQ_model_size'])

In [23]:

table_data = [AWQ_model_accuracy,T_AWQ_model, Num_parm_AWQ_model, AWQ_model_size]

headers=['1', '2', '3', 'Mean', 'STD']

tabulate(table_data, headers, tablefmt='fancy_grid')


# New column data
first_column_data =  ['AWQ_model_accuracy', 'T_AWQ_model', 'Num_parm_AWQ_model', 'AWQ_model_size']
# Add a custom index column
table_with_index = tabulate(table_data, headers=['parameters(2 layyers)'] + headers,
                            showindex=first_column_data, tablefmt="fancy_grid", numalign="center")            

# Print the extended table
print(table_with_index)

╒═════════════════════════╤═════════════╤═════════════╤═════════════╤═════════════╤═══════╕
│ parameters(2 layyers)   │      1      │      2      │      3      │    Mean     │  STD  │
╞═════════════════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═══════╡
│ AWQ_model_accuracy      │  0.697561   │  0.697561   │  0.697561   │    0.698    │   0   │
├─────────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┼───────┤
│ T_AWQ_model             │  0.986255   │   1.02617   │  0.980511   │    0.998    │ 0.025 │
├─────────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┼───────┤
│ Num_parm_AWQ_model      │    66939    │    66939    │    66939    │    66939    │   0   │
├─────────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┼───────┤
│ AWQ_model_size          │ 2.14205e+06 │ 2.14205e+06 │ 2.14205e+06 │ 2.14205e+06 │   0   │
╘═════════════════════════╧═════════════╧═════════════╧═════════════╧═══════════